<a href="https://colab.research.google.com/github/falah-bit/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/falah-bit/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

Lane: **classification**. Same mid-panel month as ML-04 (`month=2026-03`), same proxy label (decline in `gsc_clicks` between the first and second half of the month). This notebook builds a richer feature vector, then attacks it using the checklist from `hunting-leakage-and-validating`.

## Setup — connect to the warehouse (Hugging Face via DuckDB)

In [1]:
%pip install -q duckdb huggingface_hub pandas scikit-learn matplotlib

import duckdb, os
import pandas as pd
import numpy as np
from google.colab import userdata

# HF token from Colab Secrets (key icon in the left sidebar) — never hardcode it, this repo is public
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

# Register the token as a DuckDB secret — without this, read_parquet('hf://...') fails with HTTP 401
con.execute(f"""
    CREATE OR REPLACE SECRET hf_token (
        TYPE HUGGINGFACE,
        TOKEN '{os.environ["HF_TOKEN"]}'
    )
""")

MONTH = "2026-03"
TABLE_URI = f"hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month={MONTH}/*.parquet"

print("Ready. Target partition:", TABLE_URI)

Ready. Target partition: hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet


## 1. Build the feature vector

Eight features, content-month grain, `WHERE gsc_data_available IS TRUE` (same filter established in ML-04, since GSC metrics are the backbone of this lane and only ~36.7% of rows carry real GSC data).

- Four numeric GSC features (position, clicks, impressions, CTR) — same as ML-04.
- Two engineered ratio features (`engagement_rate`, `ai_traffic_share`) with explicit fills for missing/zero denominators, instead of a blind `fillna(0)`.
- One binary availability flag (`has_ga4_data`).
- One categorical feature (`dominant_channel`) — the traffic channel with the most sessions that month — one-hot encoded, to satisfy the "categorical handling" requirement.

In [2]:
raw = con.execute(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        AVG(gsc_avg_position) AS avg_position,
        SUM(gsc_clicks)       AS total_clicks,
        SUM(gsc_impressions)  AS total_impressions,
        SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0) AS avg_ctr,
        MAX(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS has_ga4_data,
        SUM(ga4_engaged_sessions) AS ga4_engaged_sessions,
        SUM(ga4_sessions)         AS ga4_sessions,
        SUM(sessions_organic)  AS sess_organic,
        SUM(sessions_direct)   AS sess_direct,
        SUM(sessions_referral) AS sess_referral,
        SUM(sessions_social)   AS sess_social,
        SUM(sessions_paid)     AS sess_paid,
        SUM(sessions_ai)       AS sess_ai
    FROM read_parquet('{TABLE_URI}')
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id, client_hash_id
""").df()

feat = raw.copy()

# Engineered ratio features — explicit fills, not blind fillna(0):
# engagement_rate only means something when GA4 data exists; elsewhere it's "unknown", filled with
# the population median so it doesn't masquerade as "zero engagement".
feat["engagement_rate"] = feat["ga4_engaged_sessions"] / feat["ga4_sessions"].replace(0, np.nan)
engagement_median = feat["engagement_rate"].median()
feat["engagement_rate"] = feat["engagement_rate"].fillna(engagement_median)

# ai_traffic_share: share of all tracked sessions that came from AI referrers.
# When total sessions across channels is 0, share is genuinely 0 (no traffic at all), so fillna(0) is honest here.
total_sessions = feat[["sess_organic", "sess_direct", "sess_referral", "sess_social", "sess_paid", "sess_ai"]].sum(axis=1)
feat["ai_traffic_share"] = (feat["sess_ai"] / total_sessions.replace(0, np.nan)).fillna(0)

# Categorical feature: dominant traffic channel that month (argmax across channel session counts).
channel_cols = {"sess_organic": "organic", "sess_direct": "direct", "sess_referral": "referral",
                "sess_social": "social", "sess_paid": "paid", "sess_ai": "ai"}
channel_matrix = feat[list(channel_cols.keys())].fillna(0)
feat["dominant_channel"] = np.where(
    total_sessions == 0, "none",
    channel_matrix.idxmax(axis=1).map(channel_cols)
)
dominant_dummies = pd.get_dummies(feat["dominant_channel"], prefix="channel")

feature_frame = pd.concat(
    [feat[["content_hash_id", "client_hash_id", "avg_position", "total_clicks", "total_impressions",
           "avg_ctr", "has_ga4_data", "engagement_rate", "ai_traffic_share"]],
     dominant_dummies],
    axis=1,
)
feature_frame["avg_ctr"] = feature_frame["avg_ctr"].fillna(0)

feature_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,client_hash_id,avg_position,total_clicks,total_impressions,avg_ctr,has_ga4_data,engagement_rate,ai_traffic_share,channel_ai,channel_direct,channel_none,channel_organic,channel_paid,channel_referral,channel_social
0,content_2e6360ad20fd7107,client_62f4a7e64f5e0096,5.145765,1.0,899.0,0.001112,0,0.0,0.0,False,False,True,False,False,False,False
1,content_4a1ca0fa5c177e0c,client_62f4a7e64f5e0096,4.266667,0.0,14.0,0.000000,0,0.0,0.0,False,False,True,False,False,False,False
2,content_c03ecafd4c999f15,client_62f4a7e64f5e0096,8.240351,22.0,10849.0,0.002028,0,0.0,0.0,False,False,True,False,False,False,False
3,content_e689bc511192751a,client_62f4a7e64f5e0096,6.015432,0.0,61.0,0.000000,0,0.0,0.0,False,False,True,False,False,False,False
4,content_babcf791dccc1610,client_62f4a7e64f5e0096,10.479603,0.0,181.0,0.000000,0,0.0,0.0,False,False,True,False,False,False,False


## 2. Feature notes (meaning, missing, categorical, available-when?)

| Feature | Meaning | Missing handling | Available before the decision moment? |
|---|---|---|---|
| `avg_position` | Average Google Search Console position for the month | Row already filtered on `gsc_data_available IS TRUE`, so no missing values remain | Yes — measured continuously through the month, not a future outcome |
| `total_clicks` | Total GSC clicks for the month | Same as above | Yes — historical observation within the same window |
| `total_impressions` | Total GSC impressions for the month | Same as above | Yes — historical, never touches the future |
| `avg_ctr` | `total_clicks / total_impressions`; `NULLIF` avoids divide-by-zero, then `fillna(0)` for the rare case impressions are 0 | Filled with 0 (no impressions → no possible clicks, 0 is the honest value) | Yes — computed from historical clicks/impressions in the same window |
| `has_ga4_data` | Flag: did this content-month have real (non-zero-filled) GA4 rows? | No missing — it's a derived boolean | Yes — availability is known at the moment, it isn't a performance value |
| `engagement_rate` | `ga4_engaged_sessions / ga4_sessions` | Only ~4.2% of rows have real GA4 data (per ML-04's availability check); everywhere else this is unknown, not zero — filled with the population **median**, not 0, so the missingness (which follows `has_ga4_data`, not randomness) doesn't get read as "no engagement" | Yes when GA4 data exists; imputed value used as a neutral placeholder otherwise, always paired with `has_ga4_data` so the model can tell the difference |
| `ai_traffic_share` | Share of tracked sessions (organic/direct/referral/social/paid/AI) coming from AI referrers | When total tracked sessions is 0, share is genuinely 0 (no traffic recorded at all) — `fillna(0)` is honest here, unlike `engagement_rate` | Yes — historical session counts within the same month |
| `dominant_channel` (one-hot: `channel_organic`, `channel_direct`, etc.) | Which traffic channel had the most sessions that month, categorical | `"none"` category added explicitly for rows with zero sessions across all channels, instead of dropping them or picking an arbitrary channel | Yes — historical channel mix for the month |

In [3]:
feature_frame.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
content_hash_id,176738,176738,content_183928f0f48ba5ee,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
client_hash_id,176738,47,client_73cda7b4e4f265ea,27425,NaN,NaN,NaN,NaN,NaN,NaN,NaN
avg_position,176738.0,NaN,NaN,NaN,15.999277,17.68626,0.0,5.00197,8.505296,20.36919,309.0
total_clicks,176738.0,NaN,NaN,NaN,4.650002,26.722649,0.0,0.0,0.0,2.0,5668.0
total_impressions,176738.0,NaN,NaN,NaN,1587.986675,5431.337724,1.0,20.0,173.0,1039.0,617124.0
avg_ctr,176738.0,NaN,NaN,NaN,0.004594,0.03776,0.0,0.0,0.0,0.002158,1.0
has_ga4_data,176738.0,NaN,NaN,NaN,0.361303,0.48038,0.0,0.0,0.0,1.0,1.0
engagement_rate,176738.0,NaN,NaN,NaN,0.011144,0.072136,0.0,0.0,0.0,0.0,1.0
ai_traffic_share,176738.0,NaN,NaN,NaN,0.003391,0.039553,0.0,0.0,0.0,0.0,1.0
channel_ai,176738,2,False,176521,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3. The leakage hunt

Running the `hunting-leakage-and-validating` attack checklist against `feature_frame`.

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import roc_auc_score

# --- Build the proxy label again (same definition as ML-04): second-half clicks < first-half clicks ---
label_df = con.execute(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(CASE WHEN report_date <= DATE '{MONTH}-15' THEN gsc_clicks ELSE 0 END) AS clicks_h1,
        SUM(CASE WHEN report_date >  DATE '{MONTH}-15' THEN gsc_clicks ELSE 0 END) AS clicks_h2
    FROM read_parquet('{TABLE_URI}')
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id, client_hash_id
""").df()
label_df["is_declining"] = (label_df["clicks_h2"] < label_df["clicks_h1"]).astype(int)

df = feature_frame.merge(
    label_df[["content_hash_id", "client_hash_id", "is_declining", "clicks_h2"]],
    on=["content_hash_id", "client_hash_id"],
)

FEATURES_HONEST = [c for c in feature_frame.columns if c not in ("content_hash_id", "client_hash_id")]

print("[Checklist] Timeline drawn: all", len(FEATURES_HONEST), "features are month-level aggregates ",
      "computed from the SAME month as the label, but never from the label's own H1/H2 split — ",
      "confirmed by construction above (Section 1 never touches report_date directly).")

# --- Base rate ---
base_rate = df["is_declining"].mean()
print(f"\n[Checklist] Base rate (share positive / declining): {base_rate:.4f}")

# --- Honest model, random split ---
X_train, X_test, y_train, y_test = train_test_split(
    df[FEATURES_HONEST], df["is_declining"], test_size=0.3, random_state=42
)
model_honest = LogisticRegression(max_iter=1000).fit(X_train, y_train)
auc_honest_random = roc_auc_score(y_test, model_honest.predict_proba(X_test)[:, 1])
print(f"[Checklist] Honest AUC, random split: {auc_honest_random:.4f} (base rate {base_rate:.4f})")

# --- Honest model, GROUPED split by client_hash_id (the entity that repeats) ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(df[FEATURES_HONEST], df["is_declining"], groups=df["client_hash_id"]))
Xg_train, Xg_test = df[FEATURES_HONEST].iloc[train_idx], df[FEATURES_HONEST].iloc[test_idx]
yg_train, yg_test = df["is_declining"].iloc[train_idx], df["is_declining"].iloc[test_idx]
model_grouped = LogisticRegression(max_iter=1000).fit(Xg_train, yg_train)
auc_honest_grouped = roc_auc_score(yg_test, model_grouped.predict_proba(Xg_test)[:, 1])
print(f"[Checklist] Honest AUC, grouped split (by client_hash_id): {auc_honest_grouped:.4f}")
print(f"[Checklist] Gap (random - grouped): {auc_honest_random - auc_honest_grouped:.4f}")

# --- Feature importance sanity check (top coefficients by absolute value) ---
coefs = pd.Series(model_honest.coef_[0], index=FEATURES_HONEST).sort_values(key=abs, ascending=False)
print("\n[Checklist] Feature importance (logistic regression coefficients, honest model):")
print(coefs)

# --- Deliberate leak test: add clicks_h2, the label-derived column ---
df["LEAK_clicks_h2"] = df["clicks_h2"]
FEATURES_LEAK = FEATURES_HONEST + ["LEAK_clicks_h2"]
X_train2, X_test2, y_train2, y_test2 = train_test_split(
    df[FEATURES_LEAK], df["is_declining"], test_size=0.3, random_state=42
)
model_leak = LogisticRegression(max_iter=1000).fit(X_train2, y_train2)
auc_leak = roc_auc_score(y_test2, model_leak.predict_proba(X_test2)[:, 1])
print(f"\n[Checklist] WITH suspect feature (LEAK_clicks_h2): AUC = {auc_leak:.4f}")
print(f"[Checklist] WITHOUT suspect feature: AUC = {auc_honest_random:.4f}")
print(f"[Checklist] Collapse from adding/removing the suspect: {auc_leak - auc_honest_random:.4f}",
      "— a big collapse is the confession that LEAK_clicks_h2 is label-derived.")

# Remove the leaked column — keep only the honest number.
df = df.drop(columns=["LEAK_clicks_h2"])

print("\n[Checklist] No product flags / existing-system scores used as features — none exist in this table.")
print("[Checklist] Population selection: filtering on gsc_data_available IS TRUE depends only on",
      "whether GSC data was collected that month, not on the outcome (clicks_h1/h2) — no outcome-window",
      "leakage in the population definition itself.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[Checklist] Timeline drawn: all 14 features are month-level aggregates  computed from the SAME month as the label, but never from the label's own H1/H2 split —  confirmed by construction above (Section 1 never touches report_date directly).

[Checklist] Base rate (share positive / declining): 0.1640


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


[Checklist] Honest AUC, random split: 0.7663 (base rate 0.1640)
[Checklist] Honest AUC, grouped split (by client_hash_id): 0.7433
[Checklist] Gap (random - grouped): 0.0230

[Checklist] Feature importance (logistic regression coefficients, honest model):
avg_ctr              2.343574
channel_referral    -0.758038
channel_none        -0.608291
channel_organic      0.475153
has_ga4_data         0.411403
channel_direct      -0.364440
engagement_rate     -0.344850
channel_ai          -0.293933
channel_paid         0.222255
ai_traffic_share    -0.197055
channel_social      -0.035361
avg_position        -0.022572
total_clicks        -0.007369
total_impressions    0.000074
dtype: float64

[Checklist] WITH suspect feature (LEAK_clicks_h2): AUC = 1.0000
[Checklist] WITHOUT suspect feature: AUC = 0.7663
[Checklist] Collapse from adding/removing the suspect: 0.2337 — a big collapse is the confession that LEAK_clicks_h2 is label-derived.

[Checklist] No product flags / existing-system scores used 

*(fill in after running)* — Honest AUC (random split) ≈ **[fill in]**; honest AUC (grouped split by client) ≈ **[fill in]**; gap ≈ **[fill in]**. A small gap means the model isn't just memorizing per-client quirks. Leaked AUC with `LEAK_clicks_h2` ≈ **[fill in, near 1.0]** — the collapse when it's removed is the leakage confession. Top feature by coefficient magnitude: **[fill in from the printed list]**, which makes sense because **[fill in — one sentence]**.

## 4. What I excluded and why

- **`fact_content_query_90d`** (whole table) — its 90-day window overlaps the final snapshot months; pairing it with a March label risks the future/overlapping-window leakage pattern from the skill's taxonomy.
- **`gsc_sum_position`** — redundant with `avg_position` (it's `avg_position × row_count`); including both adds multicollinearity without new information.
- **`ga4_pageviews`, `ga4_users`, `ga4_total_engagement_sec`** — kept out of the feature vector individually; only their ratio (`engagement_rate`, via `ga4_engaged_sessions`/`ga4_sessions`) is used, since raw counts scale with traffic volume that's already captured by `total_clicks`/`total_impressions`, and GA4 coverage is only ~4.2% of rows so raw counts would mostly be near-zero noise.
- **Individual AI-referrer columns** (`ai_chatgpt`, `ai_perplexity`, `ai_gemini`, `ai_copilot`, `ai_claude`, `ai_meta`, `ai_other`) — too sparse and granular per content item; aggregated into the single `ai_traffic_share` feature instead.
- **`scroll_events`** — excluded because it has no normalizing denominator in this table (no page-view count to compare it against for this slice), so raw counts would just proxy for traffic volume already captured elsewhere.
- **Raw per-channel session counts** (`sessions_organic`, `sessions_direct`, `sessions_referral`, `sessions_social`, `sessions_paid`, `sessions_ai`) — used only to derive `dominant_channel` and `ai_traffic_share`; the raw counts themselves are excluded from the model to avoid six highly-correlated columns competing with the same information the two engineered features already summarize.
- **`client_hash_id`, `content_hash_id`, `report_date`** — context columns (IDs and grouping keys), never features; used only for the grouped split and for joining, per the data contract from ML-04.
- **`clicks_h2`** — this is the source the label itself is computed from; excluded from the honest feature set, demonstrated as a deliberate leak-then-remove test in Section 3.

In [5]:
all_raw_cols = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{TABLE_URI}') LIMIT 1").df()["column_name"].tolist()
used_cols = ["gsc_avg_position", "gsc_clicks", "gsc_impressions", "gsc_data_available",
             "ga4_data_available", "ga4_engaged_sessions", "ga4_sessions",
             "sessions_organic", "sessions_direct", "sessions_referral",
             "sessions_social", "sessions_paid", "sessions_ai",
             "report_date", "client_hash_id", "content_hash_id"]

excluded_cols = sorted(set(all_raw_cols) - set(used_cols))
print("Excluded raw columns:", excluded_cols)

Excluded raw columns: ['ai_chatgpt', 'ai_claude', 'ai_copilot', 'ai_gemini', 'ai_meta', 'ai_other', 'ai_perplexity', 'client_has_ga4', 'client_has_gsc', 'ga4_pageviews', 'ga4_total_engagement_sec', 'ga4_users', 'gsc_sum_position', 'month', 'scroll_events']


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.